In [ ]:
# 1. Download & extract underwater dataset
!pip install -q ultralytics huggingface_hub

import os, zipfile
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

dataset_dir = "/content/underwater_data"
os.makedirs(dataset_dir, exist_ok=True)

zip_path = hf_hub_download(
    repo_id="nithilnandakumar/Underwater-Trash-Detection-Dataset",
    filename="YOLO_format.zip",
    repo_type="dataset",
    local_dir="/content"
)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(dataset_dir)

yaml_content = f"""
path: {dataset_dir}
train: train/images
val: valid/images
test: test/images

nc: 3
names: ['Bio', 'Submersible_Gear', 'Trash']
"""

yaml_path = os.path.join(dataset_dir, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())

In [ ]:
# 2. Train YOLOv8 Nano (50 epochs with underwater optical augmentations)
model = YOLO("yolov8n.pt")

results = model.train(
    data=yaml_path,
    epochs=50,
    batch=32,
    imgsz=640,
    device=0,
    amp=True,
    project="underwater_ai",
    name="debris_v2_yolov8n",
    patience=15,
    save=True,
    val=True,
    cos_lr=True,
    close_mosaic=10,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    fliplr=0.5,
    scale=0.5
)

In [ ]:
# 3. Export & download trained weights
from google.colab import files

best_weight_path = os.path.join(results.save_dir, "weights", "best.pt")
if os.path.exists(best_weight_path):
    files.download(best_weight_path)
else:
    fallback = "/content/runs/detect/underwater_ai/debris_v2_yolov8n/weights/best.pt"
    files.download(fallback)